## 6. 计算机视觉与图像生成
本章主要介绍生成式AI在计算机视觉与图像生成领域的应用。相对来说，图像相关智能任务与模型更为复杂，最早尝试使用转换器模型处理图像并取得较好效果的模型，是由Google发布的视觉转换器（Vision Transformer，以下简称ViT）模型。它的核心思想是将图像分割为多个固定大小的块（Patch），然后将这些块平铺成序列后再输入转换器模型。

### 6.1  计算机视觉任务
计算机视觉致力于让计算机像人类一样理解和解释视觉世界，它可以从图像、视频或其它视觉数据中提取有效信息，为系统执行进一步任务提供准确的数据支撑。计算机器视觉包括图像分类、对象检测、图像分割、深度估计等多种任务，这些任务在许多应用中都起到非常关键的作用。早期与计算机视觉相关的算法或模型，一般只能处理一种类型的计算机视觉任务。但在转换器模型进入这个领域后，它的应用范围几乎扩展到了所有计算机视觉任务。

#### 6.1.1  图像分类
图像分类是计算机视觉中的一个基础问题，它与文本分类、音频分类一样，也是根据内容给对象打上类别的标签，只不过分类的对象由文本和音频变成了图像。transformers图像分类任务名称为image-classification，图像可以通过保存路径、URL地址传入流水线，也可以将图像加载进来并以Base64编码后再传入流水线：

In [ ]:
from transformers import pipeline

classifier = pipeline(task="image-classification")
preds = classifier("res/pedestrians-crosswalk.jpg")
print(*preds, sep="\n")

transformers图像分类任务的默认模型为google/vit-base-patch16-224，这就是前面提到的ViT模型的一个基础版本。224代表的是输入图像大小为224×224，而16则代表切分的块大小为16×16。ViT模型如下图所示：

![ViT模型](./images/vit.png)

#### 6.1.2  对象检测
对象检测在识别图片中物体类别的同时，还以边界框（Bounding Box）的形式标记出对象的坐标范围。对象检测在transformers中的任务名称为object-detection，通过该名称可加载相应流水线并处理对象检测任务：

In [ ]:
# 需要安装timm(Torch Image Models)
from transformers import pipeline
detector = pipeline(task="object-detection")
preds = detector("res/pedestrians-crosswalk.jpg")
print(preds)

对象检测使用的默认模型为facebook/detr-resnet-50，名称中的detr指的是检测转换器（DEtection TRansformer，以下简称DETR）模型。这是一种在2020年由Facebook发布的图像模型，它是ViT之外基于转换器架构的另一种图像模型形态。它与ViT的最主要区别在于，DETR是编码器-解码器形态的转换器模型，而ViT则是仅编码器形态的转换器模型：

![DETR模型结构](./images/deter.png)

有关DETR模型的更多介绍，参见本小节书中内容。

#### 6.1.3  图像分割
图像分割是将图像划分为若干个具有语义意义的区域或部分，以便系统做进一步的分析或处理。图像分割可以分为语义分割（Semantic Segmentation）、实例分割（Instance Segmentation）和全景分割（Panoptic Segmentation）三类。transformers库中图像分割的任务名称为image-segmentation，使用该名称同样可轻松加载图像分割流水线并执行任务：

In [ ]:
from transformers import pipeline
from PIL import Image, ImageDraw
import numpy as np
import matplotlib.pyplot as plt

# 创建分割模型的 pipeline
detector = pipeline(task="image-segmentation")
# 加载图像并进行预测
image_path = "res/pedestrians-crosswalk.jpg"
preds = detector(image_path)
print(*preds, sep="\n")

执行上述代码返回的结果是掩膜矩阵，可通过如下代码绘制掩膜：

In [ ]:
# 读取原始图像
image = Image.open(image_path)
image_copy = image.copy()  # 保留原图，避免修改
# 用于绘制掩膜的画笔
draw = ImageDraw.Draw(image_copy)

# 假设 preds 中有多个掩膜
for i, pred in enumerate(preds):
    # 获取每个 mask 和其类别
    mask = pred['mask']
    label = pred['label']
    # 将 mask 转换为 NumPy 数组
    mask_array = np.array(mask)
    # 确保掩膜是二值化的（即 0 或 1）
    mask_array = mask_array > 0.5  # 这里假设掩膜值大于 0.5 表示目标区域
    # 创建随机颜色（可以选择任何颜色）
    color = np.random.randint(0, 256, size=3).tolist()
    # 将 mask 逐个绘制在图像上
    for y in range(mask_array.shape[0]):
        for x in range(mask_array.shape[1]):
            if mask_array[y, x]:
                draw.point((x, y), fill=tuple(color))
# 保存结果
image_copy.save("res/masks_drawn_image.png")

将原始图像与掩膜相比，可以看到模型分割效果还是相当不错的：

![分割结果对比](./images/mask-deter.png)

#### 6.1.4  深度估计
深度估计是预测图像中每个像素相对拍摄源的距离，是场景理解和场景重建的重要步骤。transformers库中深度估计的任务名称为depth-estimation，通过该名称可加载默认模型执行单目深度估计任务：

In [ ]:
from transformers import pipeline

depth_estimator = pipeline(task="depth-estimation")
preds = depth_estimator("res/pedestrians-crosswalk.jpg")
print(preds)
preds["depth"].save("res/depth.jpeg")

将模型输出结果灰度图保存了下来，对比原图可以看出其深度估计效果也相当不错：

![深度估计结果](./images/depth-deter.png)

transformers库深度估计使用的默认模型为Intel/dpt-large，密集预测转换模型（Dense Prediction Transformer，以下简称DPT）是适用于密集预测任务的图像模型。DPT是基于编码器-解码器架构的图像模型，但它却不能说是编码器-解码器形态的转换器模型。下图展示了DPT的基本结构：

![DPT结构](./images/dpt.png)

DPT模型具体介绍参见本小节书中内容。

### 6.2  图像生成与扩散模型
ViT、DETR和DPT基本思想都是将二维图像分割成序列数据，这种分割图像的思想在理解图像时可以提供丰富的上下文信息，但在生成图像时就显得有些力不从心了。到本书截稿时，单纯基于转换器架构的图像模型还没有在图像生成领域取得突破。图像生成领域还是以早先诞生的一些模型为主，这其中包括变分自编码器、生成式对抗网络以及扩散模型等。尽管在模型结构上，变分自编码器、生成式对抗网络更接近转换器模型，但扩散模型却是目前图像生成领域的热点模型。

#### 6.2.1 基于扩散模型生成图像
Hugging Face为了方便用户应用扩散模型，开发了专门用于扩散模型的Python库diffusers。它与transformers库极为相似，也可以以流水线的形式加载模型并处理图像生成任务。下面的示例展示了使用diffusers生成图片的过程：

In [ ]:
import torch
from PIL import Image
from matplotlib import pyplot as plt
from diffusers import StableDiffusionPipeline
import time

start = time.time()

# 设置执行设备：mps,gpu或cpu
device = "mps" if torch.backends.mps.is_available() \
    else "cuda" if torch.cuda.is_available() else "cpu"
# 加载预训练好的stable diffusion模型
model_id = "stabilityai/stable-diffusion-2-1-base"
pipe = StableDiffusionPipeline.from_pretrained(model_id).to(device)

# 设置随机数种子，用以保证每次生成的结果相同
generator = torch.Generator(device=device).manual_seed(42)

# 执行流水线，生成图像
pipe_output = pipe(
    prompt="Oil painting of an autumn cityscape",  # 正向提示
    negative_prompt="Oversaturated, blurry, low quality", # 逆向提示
    height=480,
    width=640,  # 图像尺寸
    guidance_scale=8,  # 遵从提示的强度
    num_inference_steps=35,  # 生成图像的步数
    generator=generator,  # 使用固定的随机数
)

# 保存图片
image = pipe_output.images[0]
image.save("res/autumn_oil_painting.png")
print(time.time()-start)

代码示例使用的模型为stabilityai/stable-diffusion-2-1-base，这是Stable Diffusion模型的基础版本。该模型参数文件占用的存储空间大小约为5G，所以执行上述代码需要保证足够的磁盘和内存空间。此外，图像生成相较于之前的自然语言处理要慢得多，在执行上述代码时可能需要等待较长时间。模型最终生成的图像如下：

![秋天油画](au.png)

#### 6.2.2  去噪扩散模型
扩散模型是受物理学中非平衡态热力学（Nonequilibrium Thermodynamics）的启发，先将训练数据集中的图像一步一步模糊化，然后再反过来让模型学习如何逆向恢复图片。下图描述了扩散模型的训练过程：

![扩散模型训练过程](./images/diffusion.png)

训练过程可分为正向过程（Forward Process）和逆向过程（Reverse Process）两个步骤，正向过程是人脸变模糊的过程，也就是从X0到XT的过程；而逆向过程则是人脸由模糊变回清晰的过程，即从XT到X0的过程。正向过程通过向图像添加随机噪声实现图像模糊化，逆向过程则教会模型去除噪声恢复图像。早期扩散模型采用马尔可夫链（Markov Chain）对去噪过程进行建模，效率低速度慢；改造后的算法则以一个噪声估算模型来估算图像中的噪声，估算模型是一种被称为U-Net的卷积神经网络。这种基于U-Net神经网络做噪声估算的扩散模型，一般称为去噪扩散模型（Denoising Diffusion Probabilistic Models，以下简称DDPM），更多请参阅本小节书中内容。

#### 6.2.3  自编码器模型

包括转换器模型在内的许多机器学习模型，都采用了基于编码器-解码器的架构模式。在这些模型中，自编码器模型（Autoencoder）应该是最早尝试这一架构模式的模型之一。扩散模型中使用噪声估算模型U-Net就是自编码器模型的一种变体，下节将要介绍的变分自编码器也同样基于自编码器模型。自编码器模型如图所示：

![自编码器模型](./images/autoencoder.png)

居于中间的低维空间被称为隐空间或潜空间（Latent Space），具体的一个低维抽象实例是隐变量或潜变量（Latent Variable）。

#### 6.2.4  U-Net噪声估算*
U-Net模型由下采样（Down Sample）和上采样（Up Sample）两部分组成，在结构上形成了完美的U形结构，故而被称为U-Net：

![U-Net结构](./images/unet.png)

### 6.3  潜在扩散模型
基于U-Net的去噪扩散模型DDPM是从噪声开始生成图像的，可以认为是一种不受控的图像生成模型。通过文本描述引导模型生成图像属于受控的图像生成过程，使用的图像模型是Stable Diffusion，是一种被称为潜在扩散模型（Latent Diffusion Model，以下简称LDM）的图像生成模型。LDM是目前图像生成领域最具潜力的生成式模型之一，可以说是聚集了多种模型的精华思想。这其中不仅包括DDPM中的正向、逆向过程，还包括自编码器中潜空间的思想以及转换器模型中的交叉注意力机制等等：

![LDM结构](./images/ldm.png)

更多请参阅本小节书中内容。

### 6.4  本章小结
本章以计算机视觉和图像生成为分类标准，介绍了不少与图像相关的智能任务和模型。计算机视觉任务包括图像分类、对象检测、图像分割和深度估计等，这些任务长期以来都是CNN的天下。但在转换器模型诞生以后，一些基于转换器架构的新模型也取得了相当不错的成绩，比如本章介绍的ViT、DETR和DPT等。在这些模型中，ViT属于仅编码器形态的转换器模型，适用于图像分类这种判别式任务；而DETR则是典型的编码器-解码器形态模型，非常适合处理从图像到数值序列的生成式任务；DPT虽然是基于ViT的仅编码器模型，但它采用CNN对ViT的输出做了复杂的特征重组与融合操作，所以适用于密集预测类的任务。DPT可以认为是大号的U-Net模型，只不过模型的下采样块采用了ViT模型。在这三种模型中，ViT是纯粹的转换器模型，而DETR和DPT仍然在不同程度上依赖于CNN。
在图像生成领域的主流模型包括VAE、GAN和DM等，其中的DM及其衍生的模型变体很可能会在未来主导图像生成的发展。VAE和GAN的图像生成是不受控的，图像生成从样本空间的随机采样开始，故而生成的图像也具有一定的随机性。DM可分为DDPM和LDM两类，DDPM图像生成也不受控，而且算法效率低生成图像需要很长时间。LDM则通过引入潜空间和交叉自注意力机制，将扩散模型的图像生成提升为高效率的受控图像生成。扩散模型的训练过程可分为正向扩散和逆向去噪两个过程，逆向去噪同样采用了U-Net模型预测图像噪声。LDM之所以可以受控生成图像，就是在去噪过程中通过交叉自注意力机制，将文本、图像等生成条件引入到了U-Net中。所以LDM虽然并不是基于转换器架构的模型，但转换器架构中的自注意力机制仍然在其中扮演了重要角色。
本章为了便于读者理解扩散模型，还介绍了自编码器、CLIP等模型，但它们并非仅在图像领域使用。自编码器在数据降维、异常检测等领域都有实际应用案例，VAE实际上也是对自编码器模型的改进。CLIP模型则在多模态处理领域拥有广阔的应用前景，比如基于CLIP模型可以实现图像的文本检索等等。总的来看，本章介绍的这些模型在设计思想上都非常新颖，对于理解AI底层机制以及模型创新都非常有启发。